In [ ]:
!wget -q "https://archive.ics.uci.edu/static/public/352/online+retail.zip" -O retail.zip
!unzip -o retail.zip

Archive:  retail.zip
 extracting: Online Retail.xlsx      


In [ ]:
import os
os.listdir()


['.config', 'retail.zip', 'Online Retail.xlsx', 'sample_data']

In [ ]:
import pandas as pd
import numpy as np
df = pd.read_excel("Online Retail.xlsx")
print("Dataset shape:",df.shape)
df.head()

Dataset shape: (541909, 8)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [ ]:
#check missing values and duplicates
print("Missing values:")
print(df.isnull().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())

Missing values:
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

Duplicate rows:
5268


In [13]:
#Remove duplicates
df = df.drop_duplicates()
print("rows after removing duplicates:",len(df))

rows after removing duplicates: 536641


In [14]:
#handle missing values
df['Description'] = df['Description'].fillna('Unknown Product')
df['CustomerID'] = df['CustomerID'].fillna('Unknown')

print("Missing values after cleaning:")
print(df.isnull().sum())


Missing values after cleaning:
InvoiceNo      0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
CustomerID     0
Country        0
dtype: int64


In [15]:
#convert columns to correct data
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['Quantity'] = pd.to_numeric(df['Quantity'], errors='coerce')
df['UnitPrice'] = pd.to_numeric(df['UnitPrice'], errors='coerce')

print("Data types after conversion:")
print(df.dtypes)

Data types after conversion:
InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
UnitPrice             float64
CustomerID             object
Country                object
dtype: object


In [16]:
#create calculated field
df['Revenue'] = df['Quantity'] * df['UnitPrice']

df['Year'] = df['InvoiceDate'].dt.year
df['Month'] = df['InvoiceDate'].dt.month
df['Month_Name'] = df['InvoiceDate'].dt.strftime('%B')
df['Year_Month'] = df['InvoiceDate'].dt.to_period('M').astype(str)

# Identify cancelled transactions
df['Cancelled'] = df['InvoiceNo'].astype(str).str.startswith('C')

print("New columns created:")
print(df[['Revenue', 'Year', 'Month', 'Month_Name', 'Year_Month', 'Cancelled']].head())

New columns created:
   Revenue  Year  Month Month_Name Year_Month  Cancelled
0    15.30  2010     12   December    2010-12      False
1    20.34  2010     12   December    2010-12      False
2    22.00  2010     12   December    2010-12      False
3    20.34  2010     12   December    2010-12      False
4    20.34  2010     12   December    2010-12      False


In [17]:
# Check negative quantities and unusual prices

print("Negative quantity records:", (df['Quantity'] < 0).sum())
print("Negative price records:", (df['UnitPrice'] < 0).sum())

print("\nCancelled transactions:", df['Cancelled'].sum())

print("\nQuantity statistics:")
print(df['Quantity'].describe())

print("\nUnit Price statistics:")
print(df['UnitPrice'].describe())

Negative quantity records: 10587
Negative price records: 2

Cancelled transactions: 9251

Quantity statistics:
count    536641.000000
mean          9.620029
std         219.130156
min      -80995.000000
25%           1.000000
50%           3.000000
75%          10.000000
max       80995.000000
Name: Quantity, dtype: float64

Unit Price statistics:
count    536641.000000
mean          4.632656
std          97.233118
min      -11062.060000
25%           1.250000
50%           2.080000
75%           4.130000
max       38970.000000
Name: UnitPrice, dtype: float64


In [18]:
#inspect records with negative UnitPrice

negative_price = df[df['UnitPrice'] < 0]

print("Negative price records:")
print(negative_price)

Negative price records:
       InvoiceNo StockCode      Description  Quantity         InvoiceDate  \
299983   A563186         B  Adjust bad debt         1 2011-08-12 14:51:00   
299984   A563187         B  Adjust bad debt         1 2011-08-12 14:52:00   

        UnitPrice CustomerID         Country   Revenue  Year  Month  \
299983  -11062.06    Unknown  United Kingdom -11062.06  2011      8   
299984  -11062.06    Unknown  United Kingdom -11062.06  2011      8   

       Month_Name Year_Month  Cancelled  
299983     August    2011-08      False  
299984     August    2011-08      False  


In [19]:
# Remove negative UnitPrice records

df = df[df['UnitPrice'] >= 0].copy()

print("Rows after removing negative price records:", len(df))
print("Negative price records remaining:", (df['UnitPrice'] < 0).sum())

Rows after removing negative price records: 536639
Negative price records remaining: 0


In [20]:
# Save the processed dataset

df.to_csv("Online_Retail_Processed.csv", index=False)

print("Processed dataset saved successfully!")
print("Final shape:", df.shape)

Processed dataset saved successfully!
Final shape: (536639, 14)


In [21]:
from google.colab import files

files.download("Online_Retail_Processed.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [22]:
# Q4: Basic descriptive analysis

print("Total Revenue:", df['Revenue'].sum())
print("Total Units Sold:", df['Quantity'].sum())
print("Total Transactions:", df['InvoiceNo'].nunique())
print("Unique Products:", df['StockCode'].nunique())
print("Unique Customers:", df['CustomerID'].nunique())
print("Unique Countries:", df['Country'].nunique())

Total Revenue: 9748131.074000003
Total Units Sold: 5162500
Total Transactions: 25898
Unique Products: 4070
Unique Customers: 4373
Unique Countries: 38


In [23]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

df['Quantity'] = pd.to_numeric(
    df['Quantity'], errors='coerce'
)

df['UnitPrice'] = pd.to_numeric(
    df['UnitPrice'], errors='coerce'
)

In [24]:
print(df.dtypes)

InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
UnitPrice             float64
CustomerID             object
Country                object
Revenue               float64
Year                    int32
Month                   int32
Month_Name             object
Year_Month             object
Cancelled                bool
dtype: object


In [25]:
df['Revenue'] = df['Quantity'] * df['UnitPrice']

In [26]:
df['Year'] = df['InvoiceDate'].dt.year

df['Month'] = df['InvoiceDate'].dt.month

df['Month_Name'] = df['InvoiceDate'].dt.strftime('%B')

df['Year_Month'] = df['InvoiceDate'].dt.to_period('M').astype(str)

In [28]:
df['Cancelled'] = df['InvoiceNo'].astype(str).str.startswith('C')

In [29]:
df['Cancelled'].value_counts()

,count
Cancelled,
False,527388
True,9251


In [30]:
df[['Quantity', 'UnitPrice', 'Revenue']].describe()

,Quantity,UnitPrice,Revenue
count,536639.000000,536639.000000,536639.000000
mean,9.620061,4.673900,18.165156
std,219.130564,94.857114,380.055489
min,-80995.000000,0.000000,-168469.600000
25%,1.000000,1.250000,3.750000
50%,3.000000,2.080000,9.870000
75%,10.000000,4.130000,17.400000
max,80995.000000,38970.000000,168469.600000


In [32]:
total_revenue = df['Revenue'].sum()

print("Total Revenue:", total_revenue)

Total Revenue: 9748131.074000003


In [31]:
df.to_excel(
    "Processed_Online_Retail.xlsx",
    sheet_name="Processed Data",
    index=False
)

In [33]:
top_products = (
    df.groupby('Description')['Revenue']
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

print(top_products)

Description
DOTCOM POSTAGE                        206245.48
REGENCY CAKESTAND 3 TIER              164459.49
WHITE HANGING HEART T-LIGHT HOLDER     99612.42
PARTY BUNTING                          98243.88
JUMBO BAG RED RETROSPOT                92175.79
RABBIT NIGHT LIGHT                     66661.63
POSTAGE                                66230.64
PAPER CHAIN KIT 50'S CHRISTMAS         63715.24
ASSORTED COLOUR BIRD ORNAMENT          58792.42
CHILLI LIGHTS                          53746.66
Name: Revenue, dtype: float64


In [34]:
top_countries = (
    df.groupby('Country')['Revenue']
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

print(top_countries)

Country
United Kingdom    8189252.304
Netherlands        284661.540
EIRE               262993.380
Germany            221509.470
France             197317.110
Australia          137009.770
Switzerland         56363.050
Spain               54756.030
Belgium             40910.960
Sweden              36585.410
Name: Revenue, dtype: float64


In [35]:
monthly_revenue = (
    df.groupby('Year_Month')['Revenue']
    .sum()
    .sort_index()
)

print(monthly_revenue)

Year_Month
2010-12     746723.610
2011-01     558448.560
2011-02     497026.410
2011-03     682013.980
2011-04     492367.841
2011-05     722094.100
2011-06     689977.230
2011-07     680156.991
2011-08     703510.580
2011-09    1017596.682
2011-10    1069368.230
2011-11    1456145.800
2011-12     432701.060
Name: Revenue, dtype: float64


In [36]:
top_customers = (
    df.groupby('CustomerID')['Revenue']
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

print(top_customers)

CustomerID
Unknown    1469611.65
14646.0     279489.02
18102.0     256438.49
17450.0     187322.17
14911.0     132458.73
12415.0     123725.45
14156.0     113214.59
17511.0      88125.38
16684.0      65892.08
13694.0      62690.54
Name: Revenue, dtype: float64


In [37]:
cancelled = df[df['Cancelled'] == True]

print("Cancelled transactions:", len(cancelled))

print("Cancelled revenue:", cancelled['Revenue'].sum())

Cancelled transactions: 9251
Cancelled revenue: -893979.7300000001


In [38]:
cancelled = df[df['Cancelled'] == True]

print(cancelled[['InvoiceNo',
                 'Quantity',
                 'UnitPrice',
                 'Revenue']].head())

    InvoiceNo  Quantity  UnitPrice  Revenue
141   C536379        -1      27.50   -27.50
154   C536383        -1       4.65    -4.65
235   C536391       -12       1.65   -19.80
236   C536391       -24       0.29    -6.96
237   C536391       -24       0.29    -6.96


In [39]:
print("Cancelled rows:", len(cancelled))

print("Cancelled quantity:",
      cancelled['Quantity'].sum())

print("Cancelled revenue:",
      cancelled['Revenue'].sum())

Cancelled rows: 9251
Cancelled quantity: -275560
Cancelled revenue: -893979.7300000001
